In [1]:
%run setup.py
 
import polars as pl
from configs import utils

In [2]:
lf_dogmove = utils.load_parquet(
    "../../data/processed", "DogMoveData_Windowed_Denoised.parquet", 
    utils.dog_move_data_widowed_schema
)

lf_doginfo = utils.load_large_data("../../data/raw/", "DogInfo.csv")

In [ ]:
lf_dogmove = lf_dogmove.filter(pl.col("n_samples") == 200)


lf_joined = lf_dogmove.join(
    lf_doginfo.select(["DogID", "Age months", "Weight"]),
    on="DogID",
    how="left"
)


In [8]:
PURITY_THRESHOLD = 0.75

lf_dogmove_with_label = lf_joined.with_columns([
    pl.col("Behavior_1").list.eval(
        pl.element().mode().first()
    ).list.first().alias("label"),
    
    pl.struct([
        pl.col("Behavior_1").list.eval(pl.element().mode().first()).list.first().alias("mode_value"),
        pl.col("Behavior_1")
    ]).map_elements(
        lambda x: (x["Behavior_1"].count(x["mode_value"]) / len(x["Behavior_1"])) if len(x["Behavior_1"]) > 0 else 0.0,
        return_dtype=pl.Float64
    ).alias("purity")
])

lf_dogmove_filtered = lf_dogmove_with_label.filter(
    pl.col("purity") >= PURITY_THRESHOLD
)

In [14]:
columns_to_remove = ["Behavior_1", "TestNum", "purity"]

lf_final = lf_dogmove_filtered.drop(columns_to_remove)

lf_final.sink_parquet("../../models/data/DogDeepLearning.parquet")